# SeamlessM4T v2 Compression — Phase 7 & 8 Only
Minimal notebook for completing **Phase 7 (DoRA fine-tuning)** and **Phase 8 (final results)**.

Run all cells top-to-bottom on Kaggle with T4 GPU.

In [ ]:
import os, sys, subprocess, pathlib, re, glob, json, gc, copy, time, math, shutil
import warnings; warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = not ON_KAGGLE
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'

GDRIVE_MOUNT = '/content/drive/MyDrive/cse465v5'
KAGGLE_WORK  = '/kaggle/working'

WORK_DIR  = KAGGLE_WORK if ON_KAGGLE else GDRIVE_MOUNT
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
AUDIO_DIR = f'{WORK_DIR}/audio'
FIG_DIR   = f'{WORK_DIR}/figures'
MODEL_DIR = f'{WORK_DIR}/models'

GDRIVE_ROOT = 'gdrive:cse465v5'

for d in [WORK_DIR, CKPT_DIR, AUDIO_DIR, FIG_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Platform : {PLATFORM}')
print(f'Work dir : {WORK_DIR}')

In [ ]:
if ON_KAGGLE:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash',
                   shell=True, capture_output=True)
    ver = subprocess.run('rclone version', shell=True, capture_output=True, text=True)
    print(ver.stdout.split('\n')[0])
else:
    print('Colab: rclone not needed.')

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Drive mounted at {GDRIVE_MOUNT}')
else:
    print('Kaggle: skipping Drive mount.')

In [ ]:
def _get_secret(key):
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    else:
        from google.colab import userdata
        return userdata.get(key)

if ON_KAGGLE:
    RCLONE_CONF = _get_secret('RCLONE_CONF')
    raw = RCLONE_CONF.strip()
    raw = re.sub(r'\s*(\[[^\]]+\])\s*', r'\n\1\n', raw)
    raw = re.sub(r'\s+(type|scope|token|team_drive|client_id|client_secret|'
                 r'root_folder_id|service_account_file|drive_id)\s*=\s*',
                 r'\n\1 = ', raw)
    raw = raw.strip() + '\n'
    rclone_cfg = pathlib.Path.home() / '.config/rclone/rclone.conf'
    rclone_cfg.parent.mkdir(parents=True, exist_ok=True)
    rclone_cfg.write_text(raw)
    r = subprocess.run('rclone lsd gdrive:', shell=True, capture_output=True, text=True)
    print('Drive root:' if r.returncode == 0 else 'rclone FAILED:')
    print(r.stdout[:300] or r.stderr[:300])
else:
    print('Colab: using mounted Drive.')

In [ ]:
subprocess.run([
    'pip', 'install', '-q',
    'transformers', 'datasets', 'torchaudio', 'speechbrain',
    'peft', 'librosa', 'jiwer', 'evaluate', 'sacrebleu',
    'sentencepiece', 'accelerate', 'matplotlib', 'seaborn',
], check=True)
print('All packages installed.')

In [ ]:
import torch
import torch.nn as nn
from datetime import datetime

def _rclone_push(local_path, remote_subpath):
    if not ON_KAGGLE:
        return
    r = subprocess.run(
        f'rclone copy "{local_path}" "{GDRIVE_ROOT}/{remote_subpath}/"',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[rclone] WARNING: push failed for {local_path}: {r.stderr[:200]}')

def _rclone_pull_model(stage_name):
    if not ON_KAGGLE:
        return
    local = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(local, exist_ok=True)
    r = subprocess.run(
        f'rclone sync "{GDRIVE_ROOT}/models/{stage_name}/" "{local}/"',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'[rclone] model pull failed for {stage_name}: {r.stderr[:300]}')
    print(f'[rclone] Pulled {stage_name} -> {local}')

def save_checkpoint(state, name, step=0, keep=3):
    fname = f'{name}_step{step:06d}.pt'
    path  = f'{CKPT_DIR}/{fname}'
    torch.save(state, path)
    mb = os.path.getsize(path) / 1e6
    print(f'[ckpt] Saved {fname} ({mb:.1f} MB)')
    if ON_KAGGLE:
        _rclone_push(path, 'checkpoints')
    old = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    for f in old[:-keep]:
        if os.path.exists(f):
            os.remove(f)

def load_latest_checkpoint(name):
    files = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    if not files:
        print(f'[ckpt] No checkpoint for {name!r}')
        return None
    state = torch.load(files[-1], map_location='cpu', weights_only=False)
    print(f'[ckpt] Loaded {os.path.basename(files[-1])}')
    return state

def sync_checkpoints_from_drive():
    if ON_KAGGLE:
        print('[ckpt] Syncing checkpoints from rclone remote...')
        r = subprocess.run(
            f'rclone sync "{GDRIVE_ROOT}/checkpoints/" "{CKPT_DIR}/"',
            shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'[ckpt] WARNING: {r.stderr[:300]}')
    else:
        print(f'[ckpt] Colab: reading directly from {CKPT_DIR}')
    files = sorted(os.listdir(CKPT_DIR)) if os.path.exists(CKPT_DIR) else []
    print(f'[ckpt] {len(files)} checkpoint(s) available')
    for f in files:
        mb = os.path.getsize(f'{CKPT_DIR}/{f}') / 1e6
        print(f'  {f:<55} {mb:>7.1f} MB')

sync_checkpoints_from_drive()
print('Checkpoint helpers ready.')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt, matplotlib, seaborn as sns
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120, 'savefig.bbox': 'tight'})
sns.set_style('whitegrid')

def count_params(module):
    return sum(p.numel() for p in module.parameters()) / 1e6

def count_params_detailed(model):
    bd = {}
    for name, child in model.named_children():
        bd[name] = count_params(child)
    bd['TOTAL'] = count_params(model)
    return bd

def print_model_breakdown(model, title='Model Breakdown'):
    bd = count_params_detailed(model)
    print(f'\n--- {title} ---')
    total = bd.pop('TOTAL')
    for name, p in sorted(bd.items(), key=lambda x: -x[1]):
        pct = p / total * 100 if total > 0 else 0
        print(f'  {name:<35} {p:>8.1f}M  ({pct:>5.1f}%)')
    print(f'  {"TOTAL":<35} {total:>8.1f}M')
    print('---')
    return {**bd, 'TOTAL': total}

def gpu_mem():
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1e9
        r = torch.cuda.memory_reserved() / 1e9
        print(f'  GPU mem: {a:.2f} GB alloc / {r:.2f} GB reserved')

def save_figure(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}', dpi=150, bbox_inches='tight')
    if ON_KAGGLE:
        _rclone_push(f'{FIG_DIR}/{name}', 'figures')

import torchaudio
from IPython.display import Audio as IPAudio, display

def play(audio, sr, label=''):
    if hasattr(audio, 'numpy'): audio = audio.squeeze().numpy()
    print(f'  {label}  ({len(audio)/sr:.1f}s | sr={sr})')
    display(IPAudio(audio, rate=int(sr)))

def save_audio(audio, sr, filename, label=''):
    path = f'{AUDIO_DIR}/{filename}'
    if hasattr(audio, 'numpy'): t = audio.squeeze().unsqueeze(0).float()
    else: t = torch.tensor(audio).unsqueeze(0).float()
    torchaudio.save(path, t, sr)
    mb = os.path.getsize(path) / 1e6
    print(f'[audio] Saved {filename} ({mb:.1f} MB)')

print('Core utilities ready.')

In [ ]:
from sacrebleu.metrics import BLEU, CHRF

_bleu = BLEU(effective_order=True)
_chrf = CHRF()

def find_layers_attr(component):
    for attr in ['layers', 'layer', 'inner_layers', 'encoder_layers', 'decoder_layers']:
        if hasattr(component, attr): return attr
    return None

def compute_bleu(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _bleu.sentence_score(hyp.strip(), [ref.strip()]).score

def compute_chrf(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _chrf.sentence_score(hyp.strip(), [ref.strip()]).score

def _remap_ids_for_decode(mdl, ids):
    if hasattr(mdl, '_vocab_remap_to_old'):
        remap = mdl._vocab_remap_to_old
        ids = ids.clone()
        mask = (ids >= 0) & (ids < len(remap))
        ids[mask] = remap[ids[mask]]
    return ids

def _model_input_device(mdl):
    if hasattr(mdl, 'speech_encoder'):
        return next(mdl.speech_encoder.parameters()).device
    return next(mdl.parameters()).device

def run_s2st(mdl, wav, tgt_lang='ben'):
    """Full speech-to-speech pipeline: returns (text, waveform)."""
    inputs = processor(audio=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(_model_input_device(mdl)) for k, v in inputs.items()}
    with torch.no_grad():
        try:
            out = mdl.generate(**inputs, tgt_lang=tgt_lang,
                               return_intermediate_token_ids=True)
            text_ids = _remap_ids_for_decode(mdl, out.sequences.cpu())
            text = processor.batch_decode(text_ids, skip_special_tokens=True)[0]
            wav_out = out.waveform.cpu().numpy().squeeze() if out.waveform is not None else np.zeros(16000)
            return text, wav_out
        except RuntimeError as e:
            print(f'  [run_s2st RuntimeError] {e}')
            return '', np.zeros(16000)

def run_s2t_only(mdl, wav, tgt_lang='ben'):
    """Text-only path (bypasses T2U/vocoder). Used only for debugging."""
    inputs = processor(audio=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(_model_input_device(mdl)) for k, v in inputs.items()}
    orig_voc = mdl.vocoder
    inp_device = next(iter(inputs.values())).device
    class _NoOpVocoder(nn.Module):
        def forward(self, *args, **kwargs):
            return torch.zeros(1, 1, device=inp_device), [1]
    mdl.vocoder = _NoOpVocoder()
    try:
        with torch.no_grad():
            out = mdl.generate(**inputs, tgt_lang=tgt_lang,
                               return_intermediate_token_ids=True)
    finally:
        mdl.vocoder = orig_voc
    text_ids = _remap_ids_for_decode(mdl, out.sequences.cpu())
    return processor.batch_decode(text_ids, skip_special_tokens=True)[0]

def remap_label_ids(token_ids, mdl):
    if not hasattr(mdl, '_vocab_remap_to_old'):
        return token_ids
    remap = mdl._vocab_remap_to_old
    old_to_new = {old.item(): new for new, old in enumerate(remap)}
    if token_ids.dim() == 1:
        return torch.tensor(
            [(-100 if t in (-100, -1) else old_to_new.get(t, -100))
             for t in token_ids.tolist()],
            dtype=token_ids.dtype, device=token_ids.device)
    out = token_ids.clone()
    for bi in range(out.shape[0]):
        for j in range(out.shape[1]):
            t = out[bi, j].item()
            if t in (-100, -1): continue
            out[bi, j] = old_to_new.get(t, -100)
    return out

def run_benchmark(mdl, samples, label='model', tgt_lang='ben', save_n=4):
    """Benchmark using the FULL S2ST pipeline so BLEU/ChrF reflect audio quality."""
    print(f'\n{"="*60}\n  BENCHMARK: {label}\n  Samples: {len(samples)}  Target: {tgt_lang}\n{"="*60}\n')
    gpu_mem()
    results = []
    for i, s in enumerate(samples):
        try:
            dur = len(s['wav']) / 16000
            t0 = time.time()
            # ── KEY FIX: use full S2ST path, not S2TT-only ──
            pred_text, out_wav = run_s2st(mdl, s['wav'], tgt_lang=tgt_lang)
            elapsed = time.time() - t0
            rtf  = elapsed / dur
            bleu = compute_bleu(pred_text, s['ref'])
            chrf = compute_chrf(pred_text, s['ref'])
            print(f'  [{i+1:>2}/{len(samples)}] BLEU={bleu:5.1f} ChrF={chrf:5.1f} RTF={rtf:.3f}  id={s["id"]}')
            print(f'              pred: {pred_text[:80]}')
            if save_n > 0 and i < save_n:
                save_audio(s['wav'], mdl.config.sampling_rate, f'{label}_s{i+1}in.wav')
                play(s['wav'], mdl.config.sampling_rate, f'{label}_s{i+1}in.wav')
                save_audio(out_wav, mdl.config.sampling_rate, f'{label}_s{i+1}out.wav')
                play(out_wav, mdl.config.sampling_rate, f'{label}_s{i+1}out.wav')
            results.append(dict(id=s['id'], bleu=bleu, chrf=chrf, rtf=rtf, pred=pred_text, ref=s['ref']))
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'  [{i+1:>2}/{len(samples)}] ERROR: {e}')
            results.append(dict(id=s['id'], bleu=0, chrf=0, rtf=float('nan'), pred='', ref=s.get('ref','')))
    valid = [r for r in results if not math.isnan(r['rtf'])]
    summary = dict(label=label, n=len(valid),
        avg_bleu=float(np.mean([r['bleu'] for r in valid])) if valid else 0,
        avg_chrf=float(np.mean([r['chrf'] for r in valid])) if valid else 0,
        avg_rtf=float(np.mean([r['rtf'] for r in valid])) if valid else 0,
        params_M=count_params(mdl))
    print(f'\n  Summary: BLEU={summary["avg_bleu"]:.2f}  ChrF={summary["avg_chrf"]:.2f}'
          f'  RTF={summary["avg_rtf"]:.4f}  Params={summary["params_M"]:.1f}M\n')
    return results, summary

print('Benchmark functions ready (S2ST pipeline for scoring).')


In [ ]:
_CUSTOM_STATE_FILE = '_custom_state.pt'
_PRUNING_MANIFEST = 'pruning_manifest.pt'

def _find_layers(component):
    for attr in ['layers', 'inner_layers', 'layer']:
        mod = getattr(component, attr, None)
        if isinstance(mod, nn.ModuleList) and len(mod) > 0:
            return mod
    return None


def _get_t2u_encoder_decoder(mdl):
    """SeamlessM4Tv2: stacks live under t2u_model.model.{encoder,decoder}, not t2u_model.{encoder,decoder}."""
    t2u = getattr(mdl, 't2u_model', None)
    if t2u is None:
        return None, None
    inner = getattr(t2u, 'model', None)
    if inner is None:
        return None, None
    enc = getattr(inner, 'encoder', None)
    dec = getattr(inner, 'decoder', None)
    return enc, dec


def _infer_t2u_layer_counts_from_checkpoint_dir(model_dir):
    """
    Infer actual T2U encoder/decoder depth from saved weights (fixes legacy saves
    where config.json still had full 6+6 layers).
    Self-contained (no load_hf_weights_dict): works in minimal notebooks and any cell order.
    Returns (enc_n, dec_n) or (None, None) if not inferable.
    """
    import os
    sd = None
    safe = os.path.join(model_dir, 'model.safetensors')
    if os.path.isfile(safe):
        try:
            from safetensors.torch import load_file
            sd = load_file(safe)
        except ImportError:
            pass
    if sd is None:
        pt = os.path.join(model_dir, 'pytorch_model.bin')
        if os.path.isfile(pt):
            blob = torch.load(pt, map_location='cpu', weights_only=False)
            if isinstance(blob, dict) and 'model' in blob:
                inner = blob['model']
                if isinstance(inner, dict):
                    sd = inner
                else:
                    sd = blob
            else:
                sd = blob
    if not sd:
        return None, None
    pref_e = 't2u_model.model.encoder.layers.'
    pref_d = 't2u_model.model.decoder.layers.'
    enc_idx, dec_idx = set(), set()
    for k in sd:
        if k.startswith(pref_e):
            rest = k[len(pref_e):].split('.', 1)[0]
            if rest.isdigit():
                enc_idx.add(int(rest))
        elif k.startswith(pref_d):
            rest = k[len(pref_d):].split('.', 1)[0]
            if rest.isdigit():
                dec_idx.add(int(rest))
    enc_n = (max(enc_idx) + 1) if enc_idx else None
    dec_n = (max(dec_idx) + 1) if dec_idx else None
    return enc_n, dec_n


def _sync_config_to_architecture(mdl):
    cfg = mdl.config
    updates = {}
    def _set(key, new_val):
        if hasattr(cfg, key) and getattr(cfg, key) != new_val:
            updates[key] = (getattr(cfg, key), new_val)
            setattr(cfg, key, new_val)
    if hasattr(mdl, 'shared') and hasattr(mdl.shared, 'num_embeddings'):
        _set('vocab_size', mdl.shared.num_embeddings)
    if hasattr(mdl, 'text_decoder'):
        layers = _find_layers(mdl.text_decoder)
        if layers is not None:
            _set('decoder_layers', len(layers))
            first = layers[0]
            if hasattr(first, 'ffn') and hasattr(first.ffn, 'fc1'):
                _set('decoder_ffn_dim', first.ffn.fc1.out_features)
    if hasattr(mdl, 'text_encoder'):
        layers = _find_layers(mdl.text_encoder)
        if layers is not None:
            _set('encoder_layers', len(layers))
    if hasattr(mdl, 'speech_encoder'):
        enc = mdl.speech_encoder
        layers = None
        for parent in [enc, getattr(enc, 'encoder', None)]:
            if parent is not None:
                layers = _find_layers(parent)
                if layers is not None: break
        if layers is None:
            for _, child in enc.named_children():
                layers = _find_layers(child)
                if layers is not None: break
        if layers is not None:
            _set('speech_encoder_layers', len(layers))
            first = layers[0]
            for ffn_path in ['ffn', 'feed_forward']:
                ffn = getattr(first, ffn_path, None)
                if ffn is not None:
                    fc = getattr(ffn, 'intermediate_dense', getattr(ffn, 'fc1', None))
                    if fc is not None and hasattr(fc, 'out_features'):
                        _set('speech_encoder_intermediate_size', fc.out_features)
                    break
    # ── T2U model (v2 stacks: t2u_model.model.encoder / .decoder) ──
    t2u_enc, t2u_dec = _get_t2u_encoder_decoder(mdl)
    if t2u_enc is not None:
        layers = _find_layers(t2u_enc)
        if layers is not None:
            _set('t2u_encoder_layers', len(layers))
            first = layers[0]
            if hasattr(first, 'ffn') and hasattr(first.ffn, 'fc1'):
                _set('t2u_encoder_ffn_dim', first.ffn.fc1.out_features)
    if t2u_dec is not None:
        layers = _find_layers(t2u_dec)
        if layers is not None:
            _set('t2u_decoder_layers', len(layers))
            first = layers[0]
            if hasattr(first, 'ffn') and hasattr(first.ffn, 'fc1'):
                _set('t2u_decoder_ffn_dim', first.ffn.fc1.out_features)

    # Sub-module config uses stripped names (encoder_layers / decoder_layers)
    t2u = getattr(mdl, 't2u_model', None)
    if t2u is not None and hasattr(t2u, 'config'):
        tc = t2u.config
        if t2u_enc is not None:
            layers = _find_layers(t2u_enc)
            if layers is not None and hasattr(tc, 'encoder_layers'):
                if getattr(tc, 'encoder_layers', None) != len(layers):
                    print(f'  t2u_model.config.encoder_layers: {getattr(tc, "encoder_layers", None)} -> {len(layers)}')
                    tc.encoder_layers = len(layers)
        if t2u_dec is not None:
            layers = _find_layers(t2u_dec)
            if layers is not None and hasattr(tc, 'decoder_layers'):
                if getattr(tc, 'decoder_layers', None) != len(layers):
                    print(f'  t2u_model.config.decoder_layers: {getattr(tc, "decoder_layers", None)} -> {len(layers)}')
                    tc.decoder_layers = len(layers)

    if updates:
        for k, (old, new) in updates.items():
            print(f'  config.{k}: {old} -> {new}')
    return updates

_CUSTOM_ATTR_NAMES = ['_vocab_remap_to_old']

def _save_custom_state(mdl, path):
    state = {}
    for attr in _CUSTOM_ATTR_NAMES:
        if hasattr(mdl, attr):
            state[attr] = getattr(mdl, attr)
    if state:
        torch.save(state, os.path.join(path, _CUSTOM_STATE_FILE))

def _load_custom_state(mdl, path):
    fpath = os.path.join(path, _CUSTOM_STATE_FILE)
    if not os.path.exists(fpath): return
    state = torch.load(fpath, map_location='cpu', weights_only=False)
    for k, v in state.items():
        setattr(mdl, k, v)
    print(f'  Restored custom state: {list(state.keys())}')

def _consolidate_to_single_gpu(mdl):
    """Move model to cuda:0 if split across devices by device_map='auto'."""
    if not (hasattr(mdl, 'hf_device_map') and len(set(mdl.hf_device_map.values())) > 1):
        return mdl
    print('  Multi-device map detected, consolidating to cuda:0...')
    from accelerate.hooks import remove_hook_from_submodules
    try:
        remove_hook_from_submodules(mdl)
    except AttributeError:
        # accelerate + PeftModel: hook detach/delattr can fail on root; .to() still works
        pass
    mdl = mdl.to('cuda:0')
    if hasattr(mdl, 'hf_device_map') and isinstance(getattr(mdl, 'hf_device_map', None), dict):
        try:
            d0 = torch.device('cuda:0')
            mdl.hf_device_map = {k: d0 for k in mdl.hf_device_map}
        except Exception:
            pass
    torch.cuda.empty_cache()
    print(f'  Model now on: {next(mdl.parameters()).device}')
    return mdl

def sync_model_config(mdl):
    if hasattr(mdl, 'speech_encoder'):
        enc = mdl.speech_encoder
        parent = enc.encoder if hasattr(enc, 'encoder') else enc
        if hasattr(parent, 'layers'):
            actual = len(parent.layers)
            if hasattr(mdl.config, 'speech_encoder_layers'):
                old = mdl.config.speech_encoder_layers
                if old != actual:
                    mdl.config.speech_encoder_layers = actual
                    print(f'  [config] speech_encoder_layers: {old} -> {actual}')
            if hasattr(mdl.config, 'speech_encoder_config') and hasattr(
                    mdl.config.speech_encoder_config, 'num_hidden_layers'):
                old = mdl.config.speech_encoder_config.num_hidden_layers
                if old != actual:
                    mdl.config.speech_encoder_config.num_hidden_layers = actual
            subcfg = getattr(mdl.speech_encoder, 'config', None)
            if subcfg is not None and hasattr(subcfg, 'num_hidden_layers'):
                old2 = subcfg.num_hidden_layers
                if old2 != actual:
                    subcfg.num_hidden_layers = actual
    if hasattr(mdl, 'text_decoder'):
        dec = mdl.text_decoder
        la = find_layers_attr(dec)
        if la:
            actual = len(getattr(dec, la))
            if hasattr(mdl.config, 'decoder_layers'):
                old = mdl.config.decoder_layers
                if old != actual:
                    mdl.config.decoder_layers = actual
            if hasattr(dec, 'config') and hasattr(dec.config, 'decoder_layers'):
                dec.config.decoder_layers = actual
    # T2U model (v2: t2u_model.model.encoder / .decoder)
    if hasattr(mdl, 't2u_model'):
        t2u_enc, t2u_dec = _get_t2u_encoder_decoder(mdl)
        for sub, comp, cfg_key in [
            ('encoder', t2u_enc, 't2u_encoder_layers'),
            ('decoder', t2u_dec, 't2u_decoder_layers'),
        ]:
            if comp is None:
                continue
            la = find_layers_attr(comp)
            if la:
                actual = len(getattr(comp, la))
                if hasattr(mdl.config, cfg_key):
                    old = getattr(mdl.config, cfg_key)
                    if old != actual:
                        setattr(mdl.config, cfg_key, actual)
                        print(f'  [config] {cfg_key}: {old} -> {actual}')
        t2u = mdl.t2u_model
        if hasattr(t2u, 'config'):
            tc = t2u.config
            if t2u_enc is not None:
                la = find_layers_attr(t2u_enc)
                if la and hasattr(tc, 'encoder_layers'):
                    actual = len(getattr(t2u_enc, la))
                    old = getattr(tc, 'encoder_layers', None)
                    if old != actual:
                        tc.encoder_layers = actual
                        print(f'  [config] t2u_model.config.encoder_layers: {old} -> {actual}')
            if t2u_dec is not None:
                la = find_layers_attr(t2u_dec)
                if la and hasattr(tc, 'decoder_layers'):
                    actual = len(getattr(t2u_dec, la))
                    old = getattr(tc, 'decoder_layers', None)
                    if old != actual:
                        tc.decoder_layers = actual
                        print(f'  [config] t2u_model.config.decoder_layers: {old} -> {actual}')

    print('  [config] sync done.')

def save_model_to_drive(mdl, proc, stage_name, manifest_extra=None):
    target_dir = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(target_dir, exist_ok=True)
    print(f'[model] Saving {stage_name} -> {target_dir} ...')
    _sync_config_to_architecture(mdl)
    try: sync_model_config(mdl)
    except Exception as e: print(f'  [model] sync_model_config skipped: {e}')
    _save_custom_state(mdl, target_dir)
    man = {'stage_name': stage_name}
    if manifest_extra: man.update(manifest_extra)
    torch.save(man, os.path.join(target_dir, _PRUNING_MANIFEST))
    try: mdl.save_pretrained(target_dir, safe_serialization=True)
    except: mdl.save_pretrained(target_dir)
    proc.save_pretrained(target_dir)
    total = sum(os.path.getsize(f'{target_dir}/{f}') for f in os.listdir(target_dir)) / 1e6
    print(f'[model] Local save done. {total:.0f} MB')
    if ON_KAGGLE:
        r = subprocess.run(
            f'rclone sync "{target_dir}/" "{GDRIVE_ROOT}/models/{stage_name}/"',
            shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'[model] WARNING: rclone push failed: {r.stderr[:300]}')
        else:
            print('[model] Pushed to Drive.')

def load_model_from_drive(stage_name):
    from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor
    local = f'{MODEL_DIR}/{stage_name}'
    if ON_KAGGLE and (not os.path.exists(local) or not os.listdir(local)):
        print(f'[model] Not in local cache, pulling from remote...')
        _rclone_pull_model(stage_name)
    if not os.path.exists(local) or not os.listdir(local):
        raise RuntimeError(f'[model] Path not found or empty: {local}')
    weight_files = [f for f in os.listdir(local)
                    if f.endswith('.safetensors') or f.endswith('.bin')]
    if not weight_files:
        raise RuntimeError(f'[model] No weight files in {local}')
    print(f'[model] Loading {stage_name} from {local} ...')
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(local)
    enc_n, dec_n = _infer_t2u_layer_counts_from_checkpoint_dir(local)
    if enc_n is not None and getattr(cfg, 't2u_encoder_layers', None) != enc_n:
        print(f'  [model] Repair T2U encoder depth from weights: {cfg.t2u_encoder_layers} -> {enc_n}')
        cfg.t2u_encoder_layers = enc_n
    if dec_n is not None and getattr(cfg, 't2u_decoder_layers', None) != dec_n:
        print(f'  [model] Repair T2U decoder depth from weights: {cfg.t2u_decoder_layers} -> {dec_n}')
        cfg.t2u_decoder_layers = dec_n
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        local, config=cfg, torch_dtype=torch.float16, device_map='auto')
    _load_custom_state(mdl, local)
    proc = SeamlessM4TProcessor.from_pretrained(local)
    mdl.eval()
    return mdl, proc

print('Model I/O helpers ready.')

In [ ]:
def _load_summaries_from_drive():
    ckpt = load_latest_checkpoint('all_summaries')
    if ckpt and 'summaries' in ckpt:
        return {s['label']: s for s in ckpt['summaries']}
    return {}

ALL_SUMMARIES = _load_summaries_from_drive()
print(f'Loaded {len(ALL_SUMMARIES)} existing summaries: {list(ALL_SUMMARIES.keys())}')

def store_summary(s):
    label = s['label']
    ALL_SUMMARIES[label] = s.copy()
    ordered = list(ALL_SUMMARIES.values())
    save_checkpoint({'summaries': ordered}, name='all_summaries', step=0)
    print(f'[summary] Stored {label} ({len(ALL_SUMMARIES)} total)')

def get_summaries():
    return sorted(ALL_SUMMARIES.values(), key=lambda s: s['label'])

def plot_phase_comparison(summaries=None, save_name='phase_comparison.png'):
    data = summaries or get_summaries()
    if not data: print('No summaries yet.'); return
    labels = [s['label'] for s in data]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Compression Pipeline: Phase Comparison', fontsize=15, fontweight='bold')
    metrics = [('avg_bleu', 'BLEU (higher=better)', '#2196F3'),
               ('avg_chrf', 'ChrF (higher=better)', '#4CAF50'),
               ('avg_rtf',  'RTF (lower=faster)', '#FF9800'),
               ('params_M', 'Parameters (M)', '#9C27B0')]
    for ax, (key, title, color) in zip(axes.flat, metrics):
        vals = [s.get(key, 0) for s in data]
        bars = ax.bar(range(len(labels)), vals, color=color, alpha=0.85, edgecolor='white')
        ax.set_title(title, fontweight='bold')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{v:.1f}',
                    ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()

def plot_size_vs_quality(summaries=None, save_name='size_vs_quality.png'):
    data = summaries or get_summaries()
    if not data: return
    fig, ax = plt.subplots(figsize=(10, 7))
    params = [s['params_M'] for s in data]
    bleu = [s['avg_bleu'] for s in data]
    chrf = [s['avg_chrf'] for s in data]
    ax.scatter(params, bleu, s=120, c='#2196F3', zorder=5, label='BLEU')
    ax.scatter(params, chrf, s=120, c='#4CAF50', marker='s', zorder=5, label='ChrF')
    for i, lbl in enumerate([s['label'] for s in data]):
        ax.annotate(lbl, (params[i], bleu[i]), fontsize=7, xytext=(5,5), textcoords='offset points')
    ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Score')
    ax.set_title('Model Size vs Translation Quality', fontweight='bold')
    ax.legend(); plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()

print('Summary + plotting helpers ready.')

In [ ]:
from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor

try:
    HF_TOKEN = _get_secret('HF_TOKEN')
    from huggingface_hub import login
    login(HF_TOKEN)
    print('Logged into HuggingFace Hub.')
except Exception as e:
    print(f'HF login skipped: {e}')

MODEL_NAME = 'facebook/seamless-m4t-v2-large'

def load_base_model():
    proc = SeamlessM4TProcessor.from_pretrained(MODEL_NAME)
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
    mdl.eval()
    gpu_mem()
    return mdl, proc

print('load_base_model() ready.')

In [ ]:
LOCAL_PARQUET_CACHE = "/kaggle/working/fleurs_parquet"

import concurrent.futures

BASE_PARQUET_URL = (
    "https://huggingface.co/datasets/google/fleurs/resolve/refs%2Fconvert%2Fparquet"
)

def _list_parquet_urls(lang, split):
    """
    Discover all parquet shards for a given lang/split by probing sequentially
    until a 404 is hit. Falls back to at least returning shard 0000.
    """
    import requests

    urls = []
    i = 0
    while True:
        url = f"{BASE_PARQUET_URL}/{lang}/{split}/{i:04d}.parquet?download=true"
        try:
            r = requests.head(url, timeout=15, allow_redirects=True)
            if r.status_code == 200:
                urls.append(url)
                i += 1
            else:
                break  # 404 or anything else → no more shards
        except requests.RequestException:
            break

    if not urls:
        # fallback: blindly return shard 0 so downstream raises a clear error
        urls = [f"{BASE_PARQUET_URL}/{lang}/{split}/0000.parquet?download=true"]
        print(f"  [WARN] Could not probe shards for {lang}/{split}, falling back to 0000 only")

    print(f"  [shards] {lang}/{split}: {len(urls)} shard(s) found")
    return urls

def _download_shard(args):
    import requests
    url, dest = args
    dest = pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size > 1024 * 1024:
        return url, True, "cached"
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        try:
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk: f.write(chunk)
            if dest.stat().st_size < 1024 * 1024:
                raise RuntimeError("Downloaded file too small")
            return url, True, "downloaded"
        except Exception as e:
            if dest.exists(): dest.unlink()
            if attempt == 2: return url, False, str(e)
    return url, False, "unknown error"

def load_fleurs_parallel(src_lang, tgt_lang, split="train", n_workers=4):
    import pandas as pd
    from datasets import Dataset

    tasks = []
    shard_index = {}  # lang -> list of local dest paths

    for lang in [src_lang, tgt_lang]:
        urls = _list_parquet_urls(lang, split)
        shard_index[lang] = []
        for i, url in enumerate(urls):
            dest = f"{LOCAL_PARQUET_CACHE}/{lang}/{split}_{i:04d}.parquet"
            shard_index[lang].append(dest)
            tasks.append((url, dest))

    print(f"[Parallel] Downloading {len(tasks)} shards...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        for url, ok, msg in pool.map(_download_shard, tasks):
            print(f"  {'OK' if ok else 'FAIL'}: {msg}")

    def _load_lang(lang):
        files = sorted(f for f in shard_index[lang] if os.path.exists(f))
        if not files:
            raise FileNotFoundError(f"No cached shards for {lang}")
        df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
        print(f"  Loaded {lang}: {len(df)} rows from {len(files)} shard(s)")
        return Dataset.from_pandas(df)

    return _load_lang(src_lang), _load_lang(tgt_lang)

DRIVE_FLEURS_PATH = f'{GDRIVE_ROOT}/fleurs_parquet'

def push_fleurs_to_drive():
    if not ON_KAGGLE: return
    print(f'Pushing parquet cache to Drive...')
    subprocess.run(
        f'rclone copy "{LOCAL_PARQUET_CACHE}/" "{DRIVE_FLEURS_PATH}/" --transfers=8',
        shell=True, capture_output=True, text=True)

def load_fleurs_from_drive(src_lang, tgt_lang, split='train'):
    import pandas as pd
    from datasets import Dataset
    if not ON_KAGGLE:
        print('[Drive] rclone only on Kaggle.')
        return None, None
    print(f'[Drive] Pulling FLEURS parquet...')
    r = subprocess.run(
        f'rclone copy "{DRIVE_FLEURS_PATH}/" "{LOCAL_PARQUET_CACHE}/" --transfers=8',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        return None, None
    def _load_lang(lang):
        files = sorted(glob.glob(f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_*.parquet'))
        if not files: return None
        return Dataset.from_pandas(pd.concat([pd.read_parquet(f) for f in files], ignore_index=True))
    src_ds = _load_lang(src_lang)
    tgt_ds = _load_lang(tgt_lang)
    if src_ds and tgt_ds:
        print(f'[Dataset gdrive] Loaded: {len(src_ds)} src, {len(tgt_ds)} tgt')
    return src_ds, tgt_ds

print('FLEURS data loaders ready.')

In [ ]:
import numpy as np
import torch
import torchaudio
import io
import soundfile as sf
from datasets import load_dataset


N_EVAL = 25
TARGET_LANG = "ben"
FLEURS_SRC, FLEURS_TGT = "en_us", "bn_in"

print(f"Loading FLEURS {FLEURS_SRC}->{FLEURS_TGT} for benchmarking [test]")

ds_src, ds_tgt = load_fleurs_from_drive(FLEURS_SRC, FLEURS_TGT, split="test")

if ds_src is None or ds_tgt is None:
    print("\n[Cache miss] Downloading...")
    ds_src, ds_tgt = load_fleurs_parallel(FLEURS_SRC, FLEURS_TGT, split="test", n_workers=8)  # ← fix #1: was src_ds/tgt_ds
    push_fleurs_to_drive()

# ── fix #2: robust audio loader (parquet stores bytes, not array dicts) ──────
def _load_wav(ex):
    audio = ex["audio"]
    if isinstance(audio, dict) and "array" in audio:
        arr, sr = audio["array"], audio["sampling_rate"]
    elif isinstance(audio, dict) and "bytes" in audio:
        wav, sr = sf.read(io.BytesIO(audio["bytes"]))
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        arr = wav
    else:
        raise RuntimeError(f"Unsupported audio format: {list(audio.keys())}")
    arr = np.array(arr, dtype=np.float32)
    if sr != 16000:
        arr = torchaudio.functional.resample(
            torch.tensor(arr), sr, 16000
        ).numpy()
    return arr

# build small dictionaries only for needed items
src_by_id = {}
tgt_by_id = {}

for ex in ds_src:
    src_by_id[ex["id"]] = ex
    if len(src_by_id) >= N_EVAL * 5:
        break

for ex in ds_tgt:
    tgt_by_id[ex["id"]] = ex
    if len(tgt_by_id) >= N_EVAL * 5:
        break

common_ids = sorted(set(src_by_id) & set(tgt_by_id))[:N_EVAL]

eval_samples = []

for sid in common_ids:
    eval_samples.append(
        dict(
            id=sid,
            wav=_load_wav(src_by_id[sid]),   # ← fix #2: use robust loader
            ref=tgt_by_id[sid]["transcription"],
            en_text=src_by_id[sid]["transcription"]
        )
    )

print(f"Loaded {len(eval_samples)} eval samples.")

In [ ]:
TARGET_LANG = "ben"
FLEURS_SRC, FLEURS_TGT = "en_us", "bn_in"
print(f"Loading FLEURS {FLEURS_SRC}->{FLEURS_TGT} for fine-tuning [train]")

src_ds, tgt_ds = load_fleurs_from_drive(FLEURS_SRC, FLEURS_TGT, split="train")

if src_ds is None or tgt_ds is None:
    print("\n[Cache miss] Downloading...")
    src_ds, tgt_ds = load_fleurs_parallel(FLEURS_SRC, FLEURS_TGT, split="train", n_workers=8)
    push_fleurs_to_drive()

ft_src_map = {ex['id']: ex for ex in src_ds}
ft_tgt_map = {ex['id']: ex for ex in tgt_ds}
common_ids = sorted(set(ft_src_map) & set(ft_tgt_map))
print(f'Aligned training pairs: {len(common_ids)}')

def _load_wav(ex):
    import io, soundfile as sf, librosa
    audio = ex["audio"]
    if isinstance(audio, dict) and "array" in audio:
        arr, sr = audio["array"], audio["sampling_rate"]
    elif isinstance(audio, dict) and "bytes" in audio:
        wav, sr = sf.read(io.BytesIO(audio["bytes"]))
        if wav.ndim > 1: wav = wav.mean(axis=1)
        arr = wav
    else:
        raise RuntimeError("Unsupported audio format")
    if sr != 16000:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype("float32")

ft_samples = []
for uid in common_ids:
    s, t = ft_src_map[uid], ft_tgt_map[uid]
    ref = t.get('transcription', t.get('raw_transcription', ''))
    if not ref.strip(): continue
    ft_samples.append({
        'wav':     _load_wav(s),   # source English audio
        'tgt_wav': _load_wav(t),   # target Bengali audio (needed for unit extraction)
        'ref':     ref,
    })

print(f'Usable training samples: {len(ft_samples)}')


In [ ]:
def session_status():
    print('=' * 60)
    print(f'  Platform : {PLATFORM}   Time : {datetime.now():%Y-%m-%d %H:%M}')
    if os.path.exists(CKPT_DIR):
        local_files = [f for f in glob.glob(f'{CKPT_DIR}/**/*.pt', recursive=True) if os.path.isfile(f)]
        print(f'  Checkpoint files: {len(local_files)}')
        for f in sorted(local_files)[:20]:
            rel_path = os.path.relpath(f, CKPT_DIR)
            print(f'    {rel_path:<50} {os.path.getsize(f)/1e6:>8.1f} MB')
    if torch.cuda.is_available():
        print(f'  GPU: {torch.cuda.get_device_name(0)}')
        print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('=' * 60)

session_status()

---
# Phase 7: S2S Recovery Fine-tuning with DoRA
**Paper:** DoRA (Liu et al., ICML 2024 Oral)

Fine-tune using a **combined S2ST + T2U loss** so that *both* the text decoder
and the T2U (unit prediction) model receive gradients.

### Why the old S2TT-only loss was wrong
- `model(labels=text_labels)` only exercises the text decoder path; T2U and the vocoder are never invoked.
- After pruning, T2U is in a degraded state. Training only the text decoder leaves T2U broken → gibberish audio.

### What we fix
1. **Unit label extraction** — run `model.generate()` on Bengali target audio to obtain `unit_ids` (discrete unit sequence that T2U must predict).
2. **`compute_t2u_loss()`** — calls `model(unit_labels=unit_labels)` which triggers the T2U decoder cross-entropy.
3. **Combined loss** — `loss = 0.4 * l_s2tt + 0.6 * l_t2u` so both pathways are trained simultaneously.
4. **Benchmark fixed** — `run_benchmark` now uses the full S2ST pipeline, so BLEU/ChrF reflect audio translation quality.


In [ ]:
model_p6, processor = load_model_from_drive('phase6_t2u_iter_pruned')
print_model_breakdown(model_p6, 'Phase 6 (input to Phase 7)')
model_p6 = _consolidate_to_single_gpu(model_p6)
save_model_to_drive(model_p6, processor, 'phase6_t2u_iter_pruned')

In [ ]:
subprocess.run(['pip', 'install', '-q', 'peft>=0.10.0'], check=True)

from peft import LoraConfig, get_peft_model, TaskType

def discover_lora_targets(mdl, scope_keywords=('text_decoder', 't2u_model', 'speech_encoder')):
    found_by_scope = {}
    for name, mod in mdl.named_modules():
        if not isinstance(mod, nn.Linear): continue
        scope = next((kw for kw in scope_keywords if kw in name), None)
        if scope is None: continue
        leaf = name.split('.')[-1]
        found_by_scope.setdefault(scope, set()).add(leaf)
    print("Linear layer leaf names by scope:")
    all_leaves = set()
    for scope, leaves in sorted(found_by_scope.items()):
        print(f"  {scope}: {sorted(leaves)}")
        all_leaves |= leaves
    attn_ffn_candidates = {
        'q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2',
    }
    targets = sorted(all_leaves & attn_ffn_candidates)
    count = sum(1 for name, mod in mdl.named_modules()
                if isinstance(mod, nn.Linear)
                and name.split('.')[-1] in targets
                and any(kw in name for kw in scope_keywords))
    print(f"\nTarget modules: {targets}  ({count} Linear layers)")
    return targets

targets = discover_lora_targets(model_p6)

In [ ]:
LORA_R     = 16
LORA_ALPHA = 32
LORA_DROP  = 0.05

lora_cfg = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROP,
    bias           = 'none',
    use_dora       = True,
    target_modules = targets,
)

model_p7 = get_peft_model(model_p6, lora_cfg)
model_p7.print_trainable_parameters()

model_p7 = _consolidate_to_single_gpu(model_p7)
model_p7.train()

In [ ]:
# ── Unit label extraction ──────────────────────────────────────────────────
# SeamlessM4T generates discrete speech units (from a HuBERT/mHuBERT tokenizer)
# that the T2U decoder must predict.  We extract them from the Bengali target
# audio using the model's own generate() path, then cache to disk.

UNIT_CACHE_PATH = f'{CKPT_DIR}/unit_labels_cache.pt'

@torch.no_grad()
def extract_unit_labels_from_audio(mdl, tgt_wav, tgt_lang='ben'):
    """
    Run the model on Bengali target audio to extract the unit_ids sequence.
    These are the discrete speech units the T2U decoder is trained to predict.
    Returns a 1-D LongTensor of unit ids, or None on failure.
    """
    inputs = processor(audio=tgt_wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(_model_input_device(mdl)) for k, v in inputs.items()}
    try:
        # generate() with return_intermediate_token_ids=True returns unit_ids
        out = mdl.generate(**inputs, tgt_lang=tgt_lang,
                           return_intermediate_token_ids=True)
        if hasattr(out, 'unit_ids') and out.unit_ids is not None:
            return out.unit_ids.cpu().squeeze(0)          # (T,)
        # Fallback: some HF versions store them differently
        if hasattr(out, 'decoder_unit_sequences'):
            return out.decoder_unit_sequences.cpu().squeeze(0)
    except Exception as e:
        pass
    return None


def build_unit_label_cache(mdl, ft_samples, cache_path, tgt_lang='ben'):
    """
    Extract unit labels for every sample and save to disk.
    Skips samples where extraction fails (keeps None).
    """
    mdl.eval()
    cache = {}
    ok, failed = 0, 0
    for i, s in enumerate(ft_samples):
        uid = i   # index as key
        units = extract_unit_labels_from_audio(mdl, s['tgt_wav'], tgt_lang=tgt_lang)
        cache[uid] = units
        if units is not None:
            ok += 1
        else:
            failed += 1
        if (i + 1) % 100 == 0:
            print(f'  Extracted {i+1}/{len(ft_samples)} | ok={ok} failed={failed}')
    torch.save(cache, cache_path)
    print(f'Unit cache saved to {cache_path}  ({ok} valid, {failed} failed)')
    return cache


# Load cache if already computed, otherwise build it
if os.path.exists(UNIT_CACHE_PATH):
    unit_cache = torch.load(UNIT_CACHE_PATH, map_location='cpu', weights_only=False)
    valid_count = sum(1 for v in unit_cache.values() if v is not None)
    print(f'[unit cache] Loaded from {UNIT_CACHE_PATH}  ({valid_count}/{len(unit_cache)} valid)')
else:
    print('[unit cache] Building unit label cache (this runs model.generate() on Bengali audio)...')
    unit_cache = build_unit_label_cache(model_p7, ft_samples, UNIT_CACHE_PATH)
    if ON_KAGGLE:
        _rclone_push(UNIT_CACHE_PATH, 'checkpoints')
        print('[unit cache] Pushed cache to Drive.')

# Attach cache index to each sample for easy lookup during training
for i, s in enumerate(ft_samples):
    s['unit_cache_idx'] = i

print('Unit extraction ready.')


In [ ]:
# ── Loss functions ─────────────────────────────────────────────────────────

S2TT_WEIGHT = 0.4   # text-decoder loss weight
T2U_WEIGHT  = 0.6   # T2U-decoder loss weight (higher: audio pathway needs more recovery)

def prepare_s2tt_batch(batch, processor, device, tgt_lang, mdl):
    audios  = [s['wav'] for s in batch]
    targets = [s['ref'] for s in batch]
    audio_enc  = processor(audio=audios, sampling_rate=16000, return_tensors='pt', padding=True)
    input_feats = audio_enc['input_features'].to(device)
    attn_mask   = audio_enc['attention_mask'].to(device)
    tok      = processor.tokenizer
    text_enc = tok(text_target=targets, tgt_lang=tgt_lang, return_tensors='pt', padding=True)
    labels   = text_enc['input_ids'].to(device)
    pad = tok.pad_token_id
    if pad is not None:
        labels = labels.masked_fill(labels == pad, -100)
    labels = remap_label_ids(labels, mdl)
    return input_feats, attn_mask, labels


def compute_s2tt_loss(model, input_feats, attn_mask, labels):
    """Text-decoder cross-entropy (trains speech_encoder + text_decoder)."""
    outputs = model(input_features=input_feats, attention_mask=attn_mask,
                    labels=labels, return_dict=True)
    return outputs.loss


def compute_t2u_loss(model, input_feats, attn_mask, unit_labels_list):
    """
    T2U cross-entropy: trains the full speech_encoder -> text_decoder -> T2U pathway.

    unit_labels_list : list of 1-D LongTensors (one per sample), may contain None.
    Returns the mean T2U loss over valid samples, or None if all failed.
    """
    valid_losses = []
    for idx, unit_labels in enumerate(unit_labels_list):
        if unit_labels is None:
            continue
        # Process one sample at a time to avoid padding mismatch in unit sequences
        feats_i = input_feats[idx:idx+1]
        mask_i  = attn_mask[idx:idx+1]
        ul = unit_labels.unsqueeze(0).to(feats_i.device)   # (1, T)
        # Mask padding token (0) with -100 so it doesn't contribute to loss
        ul = ul.masked_fill(ul == 0, -100)
        try:
            out = model(input_features=feats_i, attention_mask=mask_i,
                        unit_labels=ul, return_dict=True)
            if out.loss is not None:
                valid_losses.append(out.loss)
        except Exception as e:
            pass  # skip individual failures silently
    if not valid_losses:
        return None
    return torch.stack(valid_losses).mean()


print('Loss functions ready (S2TT + T2U combined).')


In [ ]:
import torch
import random
import time
import logging
import gc as _stdlib_gc

MAX_STEPS  = 5000
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR         = 3e-4
GRAD_CLIP  = 1.0
LOG_EVERY  = 50
SAVE_EVERY = 250

trainable = [p for p in model_p7.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(trainable, lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_STEPS)

ft_ckpt = load_latest_checkpoint('phase7_ft')
start_step = 0
loss_log = []

if ft_ckpt and ft_ckpt.get('step', 0) > 0:
    start_step = ft_ckpt['step']
    loss_log = ft_ckpt.get('loss_log', [])
    ostate = ft_ckpt.get('optimizer_state') or ft_ckpt.get('opt')
    sstate = ft_ckpt.get('scheduler_state') or ft_ckpt.get('sched')
    if ostate: optimizer.load_state_dict(ostate)
    if sstate: scheduler.load_state_dict(sstate)
    print(f'Resuming from step {start_step}')
else:
    print('Starting Phase 7 from scratch.')

# Silence verbose HF transformer logs during training
_m4t_train_log = logging.getLogger(
    'transformers.models.seamless_m4t_v2.modeling_seamless_m4t_v2')
_prev_hf_level = _m4t_train_log.level
_m4t_train_log.setLevel(logging.ERROR)

try:
    model_p7.train()
    device = next(model_p7.parameters()).device
    optim_steps = start_step
    micro_step = 0
    consecutive_errors = 0
    optimizer.zero_grad()
    t0 = time.time()

    while optim_steps < MAX_STEPS:
        indices = random.sample(range(len(ft_samples)), min(BATCH_SIZE, len(ft_samples)))
        batch = [ft_samples[i] for i in indices]

        try:
            # ── S2TT loss (text decoder) ──────────────────────────────────
            input_feats, attn_mask, text_labels = prepare_s2tt_batch(
                batch, processor, device, TARGET_LANG, model_p7)

            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                l_s2tt = compute_s2tt_loss(model_p7, input_feats, attn_mask, text_labels)

            # ── T2U loss (unit decoder) ───────────────────────────────────
            unit_labels_batch = [
                unit_cache.get(s['unit_cache_idx']) for s in batch
            ]

            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                l_t2u = compute_t2u_loss(model_p7, input_feats, attn_mask,
                                         unit_labels_batch)

            # ── Combined weighted loss ────────────────────────────────────
            if l_t2u is not None:
                loss = (S2TT_WEIGHT * l_s2tt + T2U_WEIGHT * l_t2u) / GRAD_ACCUM
            else:
                # Fallback: T2U labels not available for this batch
                loss = l_s2tt / GRAD_ACCUM

            loss.backward()
            consecutive_errors = 0

        except Exception as e:
            consecutive_errors += 1
            print(f'  [ERR] Step {optim_steps}: {e}')
            if consecutive_errors > 5:
                print('CRITICAL: Too many consecutive errors, stopping.')
                break
            optimizer.zero_grad()
            continue

        loss_log.append(loss.item() * GRAD_ACCUM)

        if (micro_step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            optim_steps += 1

            if optim_steps % LOG_EVERY == 0:
                avg_loss = sum(loss_log[-LOG_EVERY:]) / LOG_EVERY
                elapsed = time.time() - t0
                print(f'Step {optim_steps}/{MAX_STEPS} | loss: {avg_loss:.4f} | elapsed: {elapsed/60:.1f}min')

            if optim_steps % SAVE_EVERY == 0:
                model_p7.save_pretrained(f'{MODEL_DIR}/phase7_dora_adapter')
                save_checkpoint(
                    dict(step=optim_steps, loss_log=loss_log,
                         optimizer_state=optimizer.state_dict(),
                         scheduler_state=scheduler.state_dict()),
                    name='phase7_ft', step=optim_steps)

        micro_step += 1

    print(f'\nTraining complete. Final step: {optim_steps}  Total time: {(time.time()-t0)/60:.1f} min')
    model_p7.save_pretrained(f'{MODEL_DIR}/phase7_dora_adapter')
    save_checkpoint(
        dict(step=optim_steps, loss_log=loss_log,
             optimizer_state=optimizer.state_dict(),
             scheduler_state=scheduler.state_dict()),
        name='phase7_ft', step=optim_steps)
finally:
    _m4t_train_log.setLevel(_prev_hf_level)


In [ ]:
import gc as _stdlib_gc

print('Merging DoRA adapters into base model...')
model_p7_merged = model_p7.merge_and_unload()
model_p7_merged.eval()
_stdlib_gc.collect(); torch.cuda.empty_cache()
print('Merge complete.')

sync_model_config(model_p7_merged)
# Peft merge can leave config.decoder_layers stale vs pruned ModuleList → bad saves / missing keys on load.
if hasattr(model_p7_merged, 'text_decoder') and getattr(model_p7_merged.text_decoder, 'layers', None) is not None:
    n = len(model_p7_merged.text_decoder.layers)
    if getattr(model_p7_merged.config, 'decoder_layers', None) != n:
        print(f'  [phase7 merge] decoder_layers {model_p7_merged.config.decoder_layers} -> {n} (ModuleList)')
        model_p7_merged.config.decoder_layers = n
if hasattr(model_p7_merged, 'speech_encoder') and getattr(model_p7_merged.speech_encoder, 'encoder', None):
    enc = model_p7_merged.speech_encoder.encoder
    if getattr(enc, 'layers', None) is not None:
        n = len(enc.layers)
        if getattr(model_p7_merged.config, 'speech_encoder_layers', None) != n:
            print(f'  [phase7 merge] speech_encoder_layers {model_p7_merged.config.speech_encoder_layers} -> {n}')
            model_p7_merged.config.speech_encoder_layers = n
t2u_e = getattr(getattr(model_p7_merged.t2u_model, 'model', None), 'encoder', None)
t2u_d = getattr(getattr(model_p7_merged.t2u_model, 'model', None), 'decoder', None)
if t2u_e is not None and getattr(t2u_e, 'layers', None) is not None:
    n = len(t2u_e.layers)
    if getattr(model_p7_merged.config, 't2u_encoder_layers', None) != n:
        print(f'  [phase7 merge] t2u_encoder_layers {model_p7_merged.config.t2u_encoder_layers} -> {n}')
        model_p7_merged.config.t2u_encoder_layers = n
if t2u_d is not None and getattr(t2u_d, 'layers', None) is not None:
    n = len(t2u_d.layers)
    if getattr(model_p7_merged.config, 't2u_decoder_layers', None) != n:
        print(f'  [phase7 merge] t2u_decoder_layers {model_p7_merged.config.t2u_decoder_layers} -> {n}')
        model_p7_merged.config.t2u_decoder_layers = n

save_model_to_drive(model_p7_merged, processor, 'phase7_dora_merged')
print_model_breakdown(model_p7_merged, 'After Phase 7: DoRA Fine-tuned & Merged')

In [ ]:
ft_ckpt = load_latest_checkpoint('phase7_ft')
if ft_ckpt and ft_ckpt.get('loss_log'):
    losses = ft_ckpt['loss_log']
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(losses, alpha=0.25, color='steelblue', lw=0.5, label='Raw')
    ema, val = [], losses[0]
    for l in losses:
        val = 0.05 * l + 0.95 * val
        ema.append(val)
    ax.plot(ema, color='steelblue', lw=2, label='EMA')
    ax.set_xlabel('Step'); ax.set_ylabel('S2TT Cross-Entropy Loss')
    ax.set_title('Phase 7: DoRA Fine-tuning Loss')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, 'phase7_loss.png')
    plt.show()

In [ ]:
p7b = load_latest_checkpoint('phase7_benchmark')
if p7b:
    p7_results, p7_summary = p7b['results'], p7b['summary']
    print(f'Loaded P7 benchmark: BLEU={p7_summary["avg_bleu"]:.2f}  '
          f'ChrF={p7_summary["avg_chrf"]:.2f}')
else:
    p7_results, p7_summary = run_benchmark(
        model_p7_merged, eval_samples, label='P7_DoRA', save_n=4)
    save_checkpoint(dict(results=p7_results, summary=p7_summary),
                    name='phase7_benchmark', step=0)

p4b = load_latest_checkpoint('phase4_benchmark')
p6b = load_latest_checkpoint('phase6_benchmark')
p4_chrf = p4b['summary']['avg_chrf'] if p4b else 0.0
p6_chrf = p6b['summary']['avg_chrf'] if p6b else 0.0
p7_chrf = p7_summary['avg_chrf']

print(f'\n{"="*55}')
print(f'  Phase 4 ChrF : {p4_chrf:.2f}')
print(f'  Phase 6 ChrF : {p6_chrf:.2f}  (drop: {p4_chrf - p6_chrf:.2f})')
print(f'  Phase 7 ChrF : {p7_chrf:.2f}  (recovery: +{p7_chrf - p6_chrf:.2f})')
print(f'{"="*55}')

store_summary(p7_summary)

---
# Phase 8: Final Results + Paper Table

In [ ]:
sc = load_latest_checkpoint('all_summaries')
if sc and 'summaries' in sc: ALL_SUMMARIES = sc['summaries']

print('\n' + '='*80)
print('  FINAL: SeamlessM4T v2 Large  Structured Compression')
print('  Task: English to Bengali Speech Translation (FLEURS test)')
print('='*80)
hdr = f'{"Phase":<25} {"Params(M)":>10} {"Delta":>8} {"BLEU":>7} {"ChrF":>7} {"RTF":>7}'
print(hdr); print('-'*len(hdr))
bp = ALL_SUMMARIES[0]['params_M'] if ALL_SUMMARIES else 2300
for s in ALL_SUMMARIES:
    d = (1 - s['params_M']/bp)*100 if bp else 0
    ds = f'-{d:.1f}%' if d > 0 else 'base'
    print(f'  {s["label"]:<23} {s["params_M"]:>8.1f}  {ds:>7}  {s["avg_bleu"]:>6.2f}  {s["avg_chrf"]:>6.2f}  {s["avg_rtf"]:>6.4f}')
print('='*80)
if len(ALL_SUMMARIES) >= 2:
    f, b = ALL_SUMMARIES[-1], ALL_SUMMARIES[0]
    print(f'  Param reduction: {(1-f["params_M"]/b["params_M"])*100:.1f}%')
    if f['avg_rtf'] > 0:
        print(f'  Speed (RTF): {b["avg_rtf"]/f["avg_rtf"]:.2f}x faster')

In [ ]:
# Fix get_summaries to handle ALL_SUMMARIES as a list (not a dict)
def get_summaries():
    return sorted(ALL_SUMMARIES, key=lambda s: s['label'])

if len(ALL_SUMMARIES) >= 2:
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('SeamlessM4T Compression Pipeline Results', fontsize=16, fontweight='bold')
    labels = [s['label'] for s in ALL_SUMMARIES]
    x = range(len(labels))

    ax1 = fig.add_subplot(2, 3, 1)
    ps = [s['params_M'] for s in ALL_SUMMARIES]
    ax1.bar(x, ps, color='#9C27B0', alpha=0.85)
    ax1.set_ylabel('Params (M)'); ax1.set_title('Model Size', fontweight='bold')
    ax1.set_xticks(list(x)); ax1.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax2 = fig.add_subplot(2, 3, 2)
    ax2.plot(list(x), [s['avg_bleu'] for s in ALL_SUMMARIES], 'o-', color='#2196F3', lw=2)
    ax2.set_ylabel('BLEU'); ax2.set_title('BLEU (higher=better)', fontweight='bold')
    ax2.set_xticks(list(x)); ax2.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax3 = fig.add_subplot(2, 3, 3)
    ax3.plot(list(x), [s['avg_chrf'] for s in ALL_SUMMARIES], 's-', color='#4CAF50', lw=2)
    ax3.set_ylabel('ChrF'); ax3.set_title('ChrF (higher=better)', fontweight='bold')
    ax3.set_xticks(list(x)); ax3.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax4 = fig.add_subplot(2, 3, 4)
    ax4.bar(list(x), [s['avg_rtf'] for s in ALL_SUMMARIES], color='#FF9800', alpha=0.85)
    ax4.set_ylabel('RTF'); ax4.set_title('RTF (lower=faster)', fontweight='bold')
    ax4.set_xticks(list(x)); ax4.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax5 = fig.add_subplot(2, 3, 5)
    ax5.scatter(ps, [s['avg_bleu'] for s in ALL_SUMMARIES], s=100, c='#2196F3', label='BLEU')
    ax5.scatter(ps, [s['avg_chrf'] for s in ALL_SUMMARIES], s=100, c='#4CAF50', marker='s', label='ChrF')
    ax5.set_xlabel('Params (M)'); ax5.set_ylabel('Score')
    ax5.set_title('Size vs Quality', fontweight='bold'); ax5.legend(fontsize=8)

    ax6 = fig.add_subplot(2, 3, 6)
    bp = ALL_SUMMARIES[0]['params_M'] or 1
    bb = ALL_SUMMARIES[0]['avg_bleu'] or 1
    bc = ALL_SUMMARIES[0]['avg_chrf'] or 1
    comp = [(1-s['params_M']/bp)*100 for s in ALL_SUMMARIES]
    ax6.plot(comp, [s['avg_bleu']/bb*100 for s in ALL_SUMMARIES], 'o-', color='#2196F3', label='BLEU %')
    ax6.plot(comp, [s['avg_chrf']/bc*100 for s in ALL_SUMMARIES], 's-', color='#4CAF50', label='ChrF %')
    ax6.axhline(y=90, color='gray', ls='--', alpha=0.5)
    ax6.set_xlabel('Compression %'); ax6.set_ylabel('Quality Retention %')
    ax6.set_title('Compression vs Quality', fontweight='bold'); ax6.legend(fontsize=8)

    plt.tight_layout()
    save_figure(fig, 'final_comprehensive.png')
    plt.show()

plot_phase_comparison()
plot_size_vs_quality()

In [ ]:
print('\nDone. All results saved to Drive.')
session_status()